# Notebook 1 — Data Ingestion & ETL
## E-Redes Hourly Electricity Consumption by Postal Code

---

## Project Overview

### What problem are we solving?
Portugal's electricity distribution network, operated by **E-REDES**, generates millions of hourly smart meter readings across thousands of postal codes. The goal of this project is to build a scalable data pipeline that:
- Ingests and cleans raw consumption data at scale
- Enriches it with geographic context (municipality-level mapping)
- Enables downstream analytics, forecasting, and anomaly detection

### Why does this problem matter?
Grid operators need to understand *when* and *where* demand is highest to:
- Plan infrastructure investment
- Prevent outages during peak demand
- Integrate renewable energy sources effectively
- Reduce operational costs through smarter load balancing

### Why Spark?
Smart meters generate data **continuously**, every hour, across the entire country. At national scale this is a classic Big Data workload:

| Dimension | Detail |
|-----------|--------|
| **Volume** | Millions of hourly rows  |
| **Velocity** | New readings arrive every hour from thousands of meters |
| **Variety** | Mix of directly metered and statistically profiled values |
| **Veracity** | Some readings are estimated; requires quality validation at ingestion |

Spark allows us to process, apply transformations across distributed partitions, and persist results in an analytics-optimised format (Parquet) — all without loading the full dataset into a single machine's memory.

### Notebook roadmap
1. Environment setup & SparkSession initialisation
2. Load raw E-Redes CSVs into a unified Spark DataFrame
3. Schema inspection and column standardisation
4. Data quality checks (nulls, negatives, duplicates)
5. RDD demonstration — custom aggregation by postal prefix
6. Geographic enrichment — spatial join of postal codes to municipalities
7. Persist cleaned data as partitioned Parquet

---

### Dataset Description

**Source:** E-REDES – Distribuição de Eletricidade, Open Data Portal  
**License:** CC BY 4.0  
**Citation:** E-REDES – Distribuição de Eletricidade, *"E-REDES Open Data Portal"*. [Online] Available at https://e-redes.opendatasoft.com/pages/homepage/

The dataset provides **hourly electricity consumption aggregated by postal code group** across mainland Portugal. Values are derived from smart meter load diagrams. Where meters are unavailable or experience communication failures, E-REDES applies a statistical **consumption profiling methodology** to estimate values — see the [profiling methodology document](https://www.e-redes.pt/sites/eredes/files/2021-08/DocMetodologico_Perfis.pdf).

**Key limitations to be aware of:**
- **Approximate values** — all consumption figures are approximations subject to subsequent revision by E-REDES.
- **GDPR aggregation** — low-population postal codes are grouped into the `OUTROS` category to protect individual privacy.
- **Mixed data quality** — some records are metered, others are statistically profiled. The methodology document distinguishes these but the raw export does not flag them individually.

---
## 1. Environment Setup


In [1]:
from pathlib import Path
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Running in Google Colab")

    PROJECT_ROOT = Path(
        "/content/drive/MyDrive/e_redes_v2/e_redes"
    )

    sys.path.append(str(PROJECT_ROOT))

except ImportError:
    print("Not running in Google Colab")

    PROJECT_ROOT = Path.cwd().parent
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.append(str(PROJECT_ROOT))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Running in Google Colab


In [2]:
# Install Java 17 — required for Spark
!sudo apt-get update -qq
!sudo apt-get install -y openjdk-17-jdk-headless -qq
!java -version

# Tell PySpark where Java lives
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
openjdk version "17.0.19" 2026-04-21
OpenJDK Runtime Environment (build 17.0.19+10-1-22.04.2-Ubuntu)
OpenJDK 64-Bit Server VM (build 17.0.19+10-1-22.04.2-Ubuntu, mixed mode, sharing)


In [3]:
import pyspark
from pyspark.sql import SparkSession, Row
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, DateType
from pyspark.sql.types import *

from utils.data_prep import (
    load_and_reproject,
    compute_intersection_weights,
    distribute_consumption
)
import re
import pandas as pd

print(f"PySpark version: {pyspark.__version__}")
print(f"Python version:  {sys.version}")

PySpark version: 4.0.2
Python version:  3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [4]:

# SparkSession configuration
#
# spark.sql.shuffle.partitions=6 — reduces the default 200 shuffle partitions
# to 6, which is appropriate for a local single-machine setup. In production
# on a multi-node cluster this would be set to 2-3x the number of CPU cores
# across all workers.
#
# spark.driver.host=127.0.0.1 — prevents hostname resolution hangs on Windows
# and some JupyterHub configurations.
#
# Running on local[*] (all available cores on a single machine).


spark = (
    SparkSession.builder
        .master("local[*]")
        .appName("bda_eredes_ingestion")
        .config("spark.executor.memory", "4g")
        .config("spark.driver.memory", "4g")
        .config("spark.sql.shuffle.partitions", "16")
        .config("spark.sql.adaptive.enabled", "true")
        .getOrCreate()
)


spark

In [5]:
# Data directory configuration
from config import (
    DATA_EREDES_DIR,
    DATA_CP_DIR,
    DATA_MUN_DIR,
    DATA_PROCESSED,
    EREDES_CSV,
    POSTAL_SHP,
    MUNICIPALITY_JSON,
    WEIGHTS_PARQUET,
    OUTPUT_PARQUET
)

print("Paths configured:")
for name, path in [("E-Redes CSV", EREDES_CSV), ("Postal shapefile", POSTAL_SHP),
                   ("Municipalities", MUNICIPALITY_JSON)]:
    exists = "" if os.path.exists(path) else "✗ NOT FOUND"
    print(f"  {exists}  {name}: {path}")

Paths configured:
    E-Redes CSV: /content/drive/MyDrive/e_redes_v2/e_redes/data/raw/e_redes/consumos_horario_codigo_postal.csv
    Postal shapefile: /content/drive/MyDrive/e_redes_v2/e_redes/data/raw/cp/CP4_EstimativaPoligonos.shp
    Municipalities: /content/drive/MyDrive/e_redes_v2/e_redes/data/raw/municipalities/georef-portugal-concelho-millesime.shp


## 2. Load Raw E-Redes Data

Spark’s `inferSchema=True` performs schema inference by scanning the dataset, which has two main implications:

- It reads the dataset twice:
  - First pass: samples data to infer column types  
  - Second pass: loads the full dataset

- On large datasets, this results in:
  - Increased I/O cost (effectively doubling read operations)
  - Potential misclassification of data types based on sampled rows

For example:
- A postal code like `"1000-001"` is correctly inferred as a string
- However, sparse numeric columns may be inferred incorrectly as integers, leading to loss of precision or incorrect type casting

### Why define the schema explicitly

Defining the schema manually provides several advantages:

- **Single-pass reading**  
  The dataset is read only once, improving performance

- **Type safety and correctness**  
  Ensures each column has the intended data type, regardless of sample distribution

- **Fail-fast behavior**  
  If the input format changes, Spark raises an error immediately instead of silently misinterpreting data


In [6]:
schema = StructType([
    StructField("Date/Time", TimestampType(), True),
    StructField("Date", DateType(), True),
    StructField("Hour", IntegerType(), True),
    StructField("Zip Code", StringType(), True),       # always String
    StructField("Active Energy (kWh)", DoubleType(), True),
    StructField("Day of the Week", StringType(), True)
])


In [7]:
# Load all E-Redes CSV files into a single Spark DataFrame.

eredes_df = spark.read.csv(
    EREDES_CSV,
    header=True,
    schema=schema,
    sep=";"
)

print(f"Files loaded successfully.")
print(f"Total rows:    {eredes_df.count():,}")
print(f"Total columns: {len(eredes_df.columns)}")

Files loaded successfully.
Total rows:    3,727,439
Total columns: 6


---
## 3. Schema Inspection & Column Standardisation

In [8]:
# Inspect the raw schema before any transformations.
# printSchema() is lazy — it reads metadata, not data.
print("Schema:")
eredes_df.printSchema()

print("\nSample rows:")
eredes_df.show(5, truncate=False)

Schema:
root
 |-- Date/Time: timestamp (nullable = true)
 |-- Date: date (nullable = true)
 |-- Hour: integer (nullable = true)
 |-- Zip Code: string (nullable = true)
 |-- Active Energy (kWh): double (nullable = true)
 |-- Day of the Week: string (nullable = true)


Sample rows:
+-------------------+----------+----+--------+-------------------+---------------+
|Date/Time          |Date      |Hour|Zip Code|Active Energy (kWh)|Day of the Week|
+-------------------+----------+----+--------+-------------------+---------------+
|2022-11-21 11:00:00|2022-11-21|NULL|4099    |28.281             |Segunda        |
|2023-05-26 13:00:00|2023-05-26|NULL|OUTROS  |146.0601593346079  |Sexta          |
|2023-09-02 00:00:00|2023-09-02|NULL|OUTROS  |88.54380445920981  |Sábado         |
|2023-02-15 18:00:00|2023-02-15|NULL|1990    |21572.196898853515 |Quarta         |
|2023-02-24 12:00:00|2023-02-24|NULL|2300    |10530.196679363422 |Sexta          |
+-------------------+----------+----+--------+---------

In [9]:
# Column name standardisation — snake_case normalisation.
#
# The raw E-Redes column names contain spaces, mixed case, parentheses, and
# units (e.g. "Active Energy (kWh)"). These are awkward to work with in
# Spark SQL and Python

# We normalise all names to snake_case by:
#   1. Stripping leading/trailing whitespace
#   2. Converting to lowercase
#   3. Replacing spaces with underscores
#   4. Removing parentheses and special characters
#
def standardise_column_name(col_name: str) -> str:
    """Normalise a column name to snake_case, removing special characters."""
    name = col_name.strip().lower()
    name = re.sub(r'[()/%]', '', name)   # remove special chars
    name = re.sub(r'[\s/]+', '_', name)  # spaces and slashes to underscores
    name = re.sub(r'_+', '_', name)      # collapse multiple underscores
    return name.strip('_')

# col.wise operations
eredes_df = eredes_df.select([
    F.col(col).alias(standardise_column_name(col))
    for col in eredes_df.columns
])

print("Standardised column names:")
for col in eredes_df.columns:
    print(f"  {col}")

Standardised column names:
  datetime
  date
  hour
  zip_code
  active_energy_kwh
  day_of_the_week


In [10]:
# Rename key columns to clearer.
# Adjust the mapping below if your column names differ after standardisation.

# Print dtypes to verify types were inferred correctly
print("Column types after standardisation:")
for col_name, dtype in eredes_df.dtypes:
    print(f"  {col_name:<30} {dtype}")

Column types after standardisation:
  datetime                       timestamp
  date                           date
  hour                           int
  zip_code                       string
  active_energy_kwh              double
  day_of_the_week                string


---
## 4. Data Quality Checks

Before proceeding with analysis and feature engineering, it is essential to assess the quality and integrity of the dataset.

This section focuses on identifying and handling common data issues such as:

- Missing values
- Invalid or impossible measurements
- Duplicate records

To ensure computational efficiency, all checks are designed using Spark’s distributed processing model, prioritizing:

- Single-pass aggregations wherever possible
- Avoidance of repeated dataset scans
- Lazy evaluation and optimized execution plans

In [11]:
# Null value audit.

# We count nulls across every column in a single pass over the data.
# Using a list comprehension to build the aggregation avoids running a
# separate .filter().count() per column, which would trigger N separate
# Spark compiles these into ONE query plan andexecutes them in a single pass — O(1) jobs regardless of column count.
print("Null value counts per column:")
null_counts = eredes_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in eredes_df.columns
])
null_counts.show(truncate=False)

Null value counts per column:
+--------+----+-------+--------+-----------------+---------------+
|datetime|date|hour   |zip_code|active_energy_kwh|day_of_the_week|
+--------+----+-------+--------+-----------------+---------------+
|0       |0   |3727439|0       |0                |0              |
+--------+----+-------+--------+-----------------+---------------+



In [12]:
# Combined quality stats — again in a SINGLE aggregation pass.
# Merging multiple metrics into one .agg() call avoids re-scanning the data.

stats = eredes_df.agg(
    F.sum(F.col("datetime").isNull().cast("int")).alias("null_ts"),
    F.sum(F.col("active_energy_kwh").isNull().cast("int")).alias("null_energy"),
    F.sum((F.col("active_energy_kwh") < 0).cast("int")).alias("negative")
)

# returns just the first row directly
row = stats.first()

print(f"{'Null timestamps:':<35} {row.null_ts:>12,}")
print(f"{'Null energy values:':<35} {row.null_energy:>12,}")
print(f"{'Negative energy values:':<35} {row.negative:>12,}")

Null timestamps:                               0
Null energy values:                            0
Negative energy values:                       15


In [13]:
# approx_count_distinct() uses the HyperLogLog algorithm — it gives an
# approximate count in a single pass with ~2% error. Exact COUNT(DISTINCT ...)
# requires a full shuffle, which is expensive at scale.

eredes_df.select(
    F.min("datetime").alias("earliest_record"),
    F.max("datetime").alias("latest_record"),
    F.approx_count_distinct("date").alias("distinct_days"),
    F.approx_count_distinct("zip_code").alias("distinct_postal_codes")
).show(truncate=False)

+-------------------+-------------------+-------------+---------------------+
|earliest_record    |latest_record      |distinct_days|distinct_postal_codes|
+-------------------+-------------------+-------------+---------------------+
|2022-11-01 00:00:00|2023-09-30 22:00:00|341          |494                  |
+-------------------+-------------------+-------------+---------------------+



In [14]:
# Clean the dataset by removing invalid records.
#
# Filtering strategy:
#   - Drop rows where timestamp or energy is null (cannot be used in analysis)
#   - Drop rows with negative energy (physically impossible for demand)
#   - Deduplicate on (timestamp, zip_code) — the natural unique key

eredes_clean = (
    eredes_df
    .filter(F.col("datetime").isNotNull())
    .filter(F.col("active_energy_kwh").isNotNull())
    .filter(F.col("active_energy_kwh") >= 0)
    .dropDuplicates(["datetime", "zip_code"]) )

total_rows = eredes_df.count()
clean_rows = eredes_clean.count()

rows_removed = total_rows - clean_rows
print(" ")
print(f"Rows before cleaning: {total_rows:,}")
print(f"Rows after cleaning:  {clean_rows:,}")
print(f"Rows removed:         {rows_removed:,}  ({rows_removed/total_rows*100:.2f}%)")

 
Rows before cleaning: 3,727,439
Rows after cleaning:  3,726,960
Rows removed:         479  (0.01%)


---
## 5. RDD Demonstration — Consumption by Postal Prefix

Spark's RDD (Resilient Distributed Dataset) API is the low-level foundation on which DataFrames are built. While the DataFrame API is preferred for most operations (it benefits from Spark's Catalyst query optimiser), RDDs are useful for custom transformations that don't map cleanly to SQL operations.

Here we use the RDD API to aggregate total consumption by the **first 4 digits** of the postal code (the postal district), demonstrating `map`, `reduceByKey`, and `sortBy`.

In [15]:
# RDD-based aggregation: total consumption by postal district.

consumption_by_district = (
    eredes_clean
    .filter(~F.col("zip_code").isin(None, "OUTROS"))
    .withColumn("district", F.substring("zip_code", 1, 4))
    .groupBy("district")
    .agg(F.sum("active_energy_kwh").alias("total_consumption"))
    .orderBy(F.desc("total_consumption"))
    .limit(15)
)


print("Top 15 postal districts by total consumption (kWh):")
print(f"{'Postal District':<20} {'Total Consumption (kWh)':>25}")
print("-" * 47)
for district, total in consumption_by_district.take(15):
    print(f"{district:<20} {total:>25,.1f}")

Top 15 postal districts by total consumption (kWh):
Postal District        Total Consumption (kWh)
-----------------------------------------------


---
## 6. Geographic Enrichment — Spatial Join to Municipalities

The E-Redes dataset uses **postal codes** as its geographic unit. For richer analysis (comparing regions, plotting on a map, joining with demographic data) we need to map postal codes to **municipalities** (concelhos).

The challenge is that a single 4-digit postal code prefix can span multiple municipalities. We resolve this using **areal interpolation**: we compute what fraction of each postal code polygon's area falls within each municipality, then distribute consumption proportionally.

**Architecture:**
- **GeoPandas** handles the polygon intersection (small static files, ~300 municipalities × ~2800 postal codes)
- **Spark** handles the large-scale join and consumption distribution

> **Note:** GeoPandas is not big-data-safe. Its use here is justified because the polygon files are small static lookups that fit comfortably in driver memory. The resulting weights table is then joined in Spark for scalable, distributed processing.

In [16]:
# Load polygon shapefiles and reproject to a common CRS.
# This step runs entirely on the Spark driver using GeoPandas.
# Input files are static small (~10 MB each), so driver-side processing is safe.

print("Loading polygon files...")
postal_gdf       = load_and_reproject(POSTAL_SHP)
municipality_gdf = load_and_reproject(MUNICIPALITY_JSON)

print(f"Postal code polygons:   {len(postal_gdf):,} records")
print(f"Municipality polygons:  {len(municipality_gdf):,} records")
print(f"\nPostal code columns:    {postal_gdf.columns.tolist()}")
print(f"Municipality columns:   {municipality_gdf.columns.tolist()}")

Loading polygon files...
Postal code polygons:   507 records
Municipality polygons:  308 records

Postal code columns:    ['CP4', 'geometry']
Municipality columns:   ['Year', 'Official_Co', 'Official_Na', 'Official_Co', 'Official_Na', 'Official_Na', 'Official_Na', 'Iso_3166_3_', 'Type', 'geometry']


In [17]:
# Compute spatial intersection weights on the driver (GeoPandas).
# For each (postal_code, municipality) pair, compute what fraction of the
# postal code's area falls within the municipality.
# These weights will be used to distribute postal-code-level consumption
# proportionally across municipalities in the Spark join below.


print("Computing intersection weights (this may take a minute)...")

weights_pd = compute_intersection_weights(
    postal_gdf,
    municipality_gdf,
    postal_col="CP4",
    municipality_col="Official_Na"
)

print(f"\nWeights computed: {len(weights_pd):,} (postal_code, municipality) pairs")
print(f"\nSample weights:")
print(weights_pd.head(10).to_string(index=False))

Computing intersection weights (this may take a minute)...

Weights computed: 2,225 (postal_code, municipality) pairs

Sample weights:
 CP4 Official_Na Official_Na Official_Na Official_Na  weight
1000      lisboa      lisboa      lisboa      lisboa     1.0
1050      lisboa      lisboa      lisboa      lisboa     1.0
1070      lisboa      lisboa      lisboa      lisboa     1.0
1100      lisboa      lisboa      lisboa      lisboa     1.0
1150      lisboa      lisboa      lisboa      lisboa     1.0
1170      lisboa      lisboa      lisboa      lisboa     1.0
1200      lisboa      lisboa      lisboa      lisboa     1.0
1250      lisboa      lisboa      lisboa      lisboa     1.0
1300      lisboa      lisboa      lisboa      lisboa     1.0
1350      lisboa      lisboa      lisboa      lisboa     1.0


In [18]:
# Sanity check: weights for each postal code should sum to ~1.0.
# A postal code is fully covered by municipalities, so the area fractions
# must add up to 100%. Deviations > 5% indicate projection mismatches or
# polygon topology errors in the source shapefiles.

weight_check = weights_pd.groupby("CP4")["weight"].sum()
print("Weight sum statistics per postal code:")
print(weight_check.describe())

problematic = weight_check[abs(weight_check - 1.0) > 0.05]
print(f"\nPostal codes with weights not summing to 1.0 (±5%): {len(problematic)}")
if len(problematic) > 0:
    print(problematic.head(10))

Weight sum statistics per postal code:
count    507.000000
mean       0.999985
std        0.000086
min        0.999035
25%        1.000000
50%        1.000000
75%        1.000000
max        1.000000
Name: weight, dtype: float64

Postal codes with weights not summing to 1.0 (±5%): 0


In [19]:
# Convert the weights Pandas DataFrame to a Spark DataFrame and persist
# as Parquet. From this point on, all joins use Spark — big-data-safe.


# Strip any duplicate columns produced by the spatial join (GeoPandas sometimes
# copies the join key into both left and right sides of the output).
weights_pd = weights_pd.loc[:, ~weights_pd.columns.duplicated()]

# createDataFrame() from a small Pandas DataFrame is safe — the weights table
# is small (~10 k rows) and this is explicitly a driver-side → Spark operation.
weights_spark = spark.createDataFrame(weights_pd)
weights_spark.write.mode("overwrite").parquet(WEIGHTS_PARQUET)
print(f"Weights saved to {WEIGHTS_PARQUET}")

print("\nWeights schema:")
weights_spark.printSchema()
weights_spark.show(5, truncate=False)

Weights saved to /content/drive/MyDrive/e_redes_v2/e_redes/data/processed/postal_municipality_weights.parquet

Weights schema:
root
 |-- CP4: string (nullable = true)
 |-- Official_Na: string (nullable = true)
 |-- weight: double (nullable = true)

+----+-----------+------------------+
|CP4 |Official_Na|weight            |
+----+-----------+------------------+
|1000|lisboa     |0.9999999999999997|
|1050|lisboa     |0.9999999999999999|
|1070|lisboa     |1.000000000000002 |
|1100|lisboa     |1.0               |
|1150|lisboa     |0.9999999959430129|
+----+-----------+------------------+
only showing top 5 rows


In [20]:
# Distribute postal-code-level consumption to municipalities using the weights.
# distribute_consumption() performs a Spark broadcast join under the hood:
#   eredes_clean  (large)  joined with  weights_spark  (small, broadcast)
# Then it multiplies active_energy_kwh × weight to get municipality-level energy.
# This join runs fully distributed across Spark partitions.

eredes_municipal = distribute_consumption(
    eredes_clean,
    weights_spark,
    consumption_col="active_energy_kwh",
    postal_col="zip_code",
    weight_postal_col="CP4",
    municipality_col="Concelho"
)

# Cache before materialising — we run several counts and transformations below.
total = eredes_municipal.count()  # materialises cache

print(f"Total rows after distribution: {total:,}")
print("\nFinal schema after geographic enrichment:")
eredes_municipal.printSchema()
print("\nSample rows:")
eredes_municipal.show(5, truncate=False)

Original total consumption:    37,519,112,841.6 kWh
Distributed total consumption: 37,517,701,036.1 kWh
Difference: 0.0038%  (OK)
Total rows after distribution: 16,991,722

Final schema after geographic enrichment:
root
 |-- datetime: timestamp (nullable = true)
 |-- date: date (nullable = true)
 |-- hour: integer (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- active_energy_kwh: double (nullable = true)
 |-- day_of_the_week: string (nullable = true)
 |-- Official_Na: string (nullable = true)
 |-- weight: double (nullable = true)
 |-- municipality_active_energy_kwh: double (nullable = true)


Sample rows:
+-------------------+----------+----+--------+------------------+---------------+-----------+---------------------+------------------------------+
|datetime           |date      |hour|zip_code|active_energy_kwh |day_of_the_week|Official_Na|weight               |municipality_active_energy_kwh|
+-------------------+----------+----+--------+------------------+-------------

### Why we avoid `.select().distinct().count()`

Using `.select("col").distinct().count()` triggers a full shuffle operation and forces Spark to build an in-memory hash set to compute uniqueness. On large datasets or high-cardinality columns, this can lead to excessive memory usage and potential OutOfMemory errors during aggregation.

To improve efficiency, we use `countDistinct()`, which allows Spark to optimize the computation into a single aggregation stage with better memory management and reduced shuffle overhead.

This approach is more stable and scalable for distributed processing workloads.

In [21]:
# Quick sanity check: how many distinct municipalities did we map to?
# And how many postal codes remained unmatched (Concelho = null)?


matched          = eredes_municipal.filter(F.col("Official_Na").isNotNull()).count()
unmatched        = total - matched
distinct_munic = eredes_municipal.agg(
    F.countDistinct("Official_Na")
).first()[0]

print(f"Total rows after enrichment: {total:,}")
print(f"Matched to a municipality:   {matched:,}  ({matched/total*100:.1f}%)")
print(f"Unmatched (OUTROS + gaps):   {unmatched:,}  ({unmatched/total*100:.1f}%)")
print(f"Distinct municipalities:     {distinct_munic:,}")

Total rows after enrichment: 16,991,722
Matched to a municipality:   16,975,692  (99.9%)
Unmatched (OUTROS + gaps):   16,030  (0.1%)
Distinct municipalities:     278


In [22]:
# Aggregate distributed consumption to (municipality, datetime) granularity.
# A single postal code can contribute fractional energy to multiple municipalities,
# so we sum the fractional contributions per municipality per hour here.

municipality_energy_hourly = eredes_municipal \
    .filter(F.col("Official_Na").isNotNull()) \
    .groupBy("Official_Na", eredes_municipal["datetime"]) \
    .agg(F.sum("municipality_active_energy_kwh").alias("total_active_energy_kwh")) \
    .select(
        F.col("Official_Na").alias("municipality"),
        F.col("datetime"),
        F.col("total_active_energy_kwh"),
    )

In [23]:
municipality_energy_hourly.show(5, truncate=False)

+---------------+-------------------+-----------------------+
|municipality   |datetime           |total_active_energy_kwh|
+---------------+-------------------+-----------------------+
|tavira         |2023-06-27 18:00:00|19280.75037039939      |
|moura          |2023-03-07 18:00:00|5898.235524541587      |
|lisboa         |2023-01-22 09:00:00|278855.68102239433     |
|tabuaço        |2023-03-03 09:00:00|1870.9316847738567     |
|condeixa-a-nova|2023-05-03 16:00:00|5913.101846668479      |
+---------------+-------------------+-----------------------+
only showing top 5 rows


In [24]:
municipality_energy_hourly = municipality_energy_hourly.withColumnRenamed("Official_Na", "municipality")

---
## 7. Persist Cleaned Data as Parquet

The cleaned dataset is written to disk in Parquet format with partitioning and local sorting to optimize storage efficiency and future query performance.

Partitioning by `year_month` allows Spark to skip irrelevant data partitions during time-based queries, significantly reducing I/O when filtering by date ranges.

Sorting within partitions by `municipality` and `datetime` improves data locality, which helps downstream operations such as time-series analysis and aggregation by reducing scan costs and improving compression efficiency.

Parquet is used because it is a columnar storage format, which enables faster analytical queries, better compression, and efficient column pruning compared to row-based formats.

The overwrite mode ensures the output is fully reproducible and replaces any previously written dataset.

In [25]:
# Persist the cleaned, enriched DataFrame as partitioned Parquet.

municipality_energy_hourly \
    .withColumn("year_month", F.date_format("datetime", "yyyy-MM")) \
    .sortWithinPartitions("municipality", "datetime") \
    .write \
    .mode("overwrite") \
    .partitionBy("year_month") \
    .parquet(OUTPUT_PARQUET)

print(f"Data written to: {OUTPUT_PARQUET}")

Data written to: /content/drive/MyDrive/e_redes_v2/e_redes/data/processed/eredes_clean.parquet


In [26]:
# Verify the output is readable and the row count is preserved

df_verify = spark.read.parquet(OUTPUT_PARQUET)

print("Verification read from Parquet:")
print(f"  Rows:    {df_verify.count():,}")
print(f"  Columns: {len(df_verify.columns)}")
df_verify.printSchema()
df_verify.show(5, truncate=False)

spark.stop()

Verification read from Parquet:
  Rows:    2,228,170
  Columns: 4
root
 |-- municipality: string (nullable = true)
 |-- datetime: timestamp (nullable = true)
 |-- total_active_energy_kwh: double (nullable = true)
 |-- year_month: string (nullable = true)

+---------------------+-------------------+-----------------------+----------+
|municipality         |datetime           |total_active_energy_kwh|year_month|
+---------------------+-------------------+-----------------------+----------+
|almodôvar            |2022-12-06 05:00:00|1538.400824133127      |2022-12   |
|seia                 |2022-12-26 20:00:00|9164.648540869915      |2022-12   |
|reguengos de monsaraz|2022-12-17 12:00:00|3822.5263028277227     |2022-12   |
|alandroal            |2022-12-17 12:00:00|2297.1137282411455     |2022-12   |
|baião                |2022-12-22 18:00:00|6540.971600170792      |2022-12   |
+---------------------+-------------------+-----------------------+----------+
only showing top 5 rows


---
## 8. Ingestion Summary

| Step | Description | Status |
|------|-------------|--------|
| Load CSVs | All | ✓ |
| Schema standardisation | Column names normalised to snake_case | ✓ |
| Data quality | Nulls, negatives, and duplicates removed | ✓ |
| RDD demonstration | Consumption aggregated by postal district via map/reduceByKey | ✓ |
| Geographic enrichment | Postal codes mapped to municipalities via spatial intersection weights | ✓ |
| Parquet output | Clean data persisted in columnar, partitioned format | ✓ |

**Output file:** `code/data/processed/eredes_clean.parquet`  
**Next notebook:** `Notebook 2 — Exploratory Analysis & SparkSQL`

---
### Production considerations

If this pipeline were deployed in production:
- The GeoPandas intersection step would be replaced by **Apache Sedona** for distributed spatial processing
- The SparkSession would connect to a **cloud cluster** (Databricks, AWS EMR, or Google Dataproc) — the pipeline code itself requires no changes
- Schema evolution would be managed via a **schema registry** to handle E-Redes changing their column names in future exports